# SESIÓN 2: CREACIÓN Y MANIPULACIÓN DE TABLAS (PARTE II)
## Fundamentos de Programación Python para el Análisis de Datos

Inserción manual de valores en tablas usando INSERT INTO

In [ ]:
import psycopg2

# Conexión a base de datos
conn = psycopg2.connect(
    host='localhost',
    database='capacitaciones',
    user='postgres',
    password='password'
)
cur = conn.cursor()

## SLIDE 5: Introducción al INSERT INTO

In [ ]:
# Sintaxis básica de INSERT INTO
sql = """
INSERT INTO nombre_tabla (columna1, columna2, ...)
VALUES (valor1, valor2, ...);
"""
print(sql)

## SLIDE 6: Variantes de INSERT INTO

In [ ]:
# Crear tabla cursos para ejemplos
sql_create = """
CREATE TABLE IF NOT EXISTS cursos (
    id_curso SERIAL PRIMARY KEY,
    nombre_curso VARCHAR(100) NOT NULL,
    duracion INT DEFAULT 20 CHECK (duracion > 0)
)
"""

cur.execute(sql_create)
conn.commit()

# Múltiples inserciones en una sola instrucción
sql_insert_multiple = """
INSERT INTO cursos (nombre_curso, duracion)
VALUES 
    ('SQL Básico', 20), 
    ('Python para Análisis', 30), 
    ('Estadística Aplicada', 25);
"""

cur.execute(sql_insert_multiple)
conn.commit()
print("Inserciones múltiples realizadas")

## SLIDE 7: Inserciones con valores por defecto y nulos

In [ ]:
# Crear tabla participantes
sql_create_participantes = """
CREATE TABLE IF NOT EXISTS participantes (
    id_participante SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    correo VARCHAR(80) NOT NULL,
    fecha_inscripcion DATE DEFAULT CURRENT_DATE
)
"""

cur.execute(sql_create_participantes)
conn.commit()

# Inserción omitiendo columnas con DEFAULT
sql_insert_default = """
INSERT INTO participantes (nombre, correo)
VALUES ('Elena Fuentes', 'elena.fuentes@email.com');
"""

cur.execute(sql_insert_default)
conn.commit()
print("Inserción con DEFAULT realizada")

## SLIDE 8: Inserciones condicionales - Verificar existencia previa

In [ ]:
# Verificar si un correo ya existe
correo_buscar = 'elena.fuentes@email.com'

sql_verificar = f"""
SELECT COUNT(*) 
FROM participantes 
WHERE correo = '{correo_buscar}';
"""

cur.execute(sql_verificar)
resultado = cur.fetchone()
print(f"Registros encontrados con correo '{correo_buscar}': {resultado[0]}")

## SLIDE 10: Ejemplo aplicado ampliado

In [ ]:
# Crear tabla cursos completa
sql_cursos_completa = """
DROP TABLE IF EXISTS cursos CASCADE;
CREATE TABLE cursos (
    id_curso SERIAL PRIMARY KEY,
    nombre_curso VARCHAR(100) NOT NULL,
    duracion INT DEFAULT 20 CHECK (duracion > 0)
)
"""

cur.execute(sql_cursos_completa)
conn.commit()

# Inserción con valores especificados
sql_insert_1 = """
INSERT INTO cursos (nombre_curso, duracion)
VALUES ('SQL Intermedio', 25);
"""

cur.execute(sql_insert_1)
conn.commit()

# Inserción con DEFAULT
sql_insert_2 = """
INSERT INTO cursos (nombre_curso)
VALUES ('Excel para negocios');
"""

cur.execute(sql_insert_2)
conn.commit()
print("Inserciones con DEFAULT realizadas")

## SLIDE 15: Actividad guiada - Inserciones válidas e inválidas

In [ ]:
# Crear tabla para la actividad
sql_actividad = """
DROP TABLE IF EXISTS cursos CASCADE;
CREATE TABLE cursos (
    id_curso SERIAL PRIMARY KEY,
    nombre_curso VARCHAR(100) NOT NULL,
    duracion INT DEFAULT 20 CHECK (duracion > 0),
    modalidad VARCHAR(20) DEFAULT 'Online' CHECK (modalidad IN ('Online', 'Presencial'))
)
"""

cur.execute(sql_actividad)
conn.commit()
print("Tabla de actividad creada")

In [ ]:
# Inserción correcta
sql_insert_valida = """
INSERT INTO cursos (nombre_curso, duracion, modalidad)
VALUES ('Introducción a SQL', 30, 'Presencial');
"""

try:
    cur.execute(sql_insert_valida)
    conn.commit()
    print("✓ Inserción válida realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error: {e}")

In [ ]:
# Inserción con error: duración negativa (viola CHECK)
sql_insert_error_1 = """
INSERT INTO cursos (nombre_curso, duracion)
VALUES ('Curso Express', -5);
"""

try:
    cur.execute(sql_insert_error_1)
    conn.commit()
    print("✓ Inserción realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error CHECK: Duración negativa no permitida")
    print(f"  Mensaje: {e}")

In [ ]:
# Inserción con error: modalidad inválida (viola CHECK)
sql_insert_error_2 = """
INSERT INTO cursos (nombre_curso, modalidad)
VALUES ('Curso Avanzado', 'Híbrido');
"""

try:
    cur.execute(sql_insert_error_2)
    conn.commit()
    print("✓ Inserción realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error CHECK: Modalidad 'Híbrido' no permitida")
    print(f"  Valores permitidos: 'Online' o 'Presencial'")
    print(f"  Mensaje: {e}")

In [ ]:
# Inserción con valores por defecto
sql_insert_default_actividad = """
INSERT INTO cursos (nombre_curso)
VALUES ('Curso Libre');
"""

try:
    cur.execute(sql_insert_default_actividad)
    conn.commit()
    print("✓ Inserción con DEFAULT realizada")
except Exception as e:
    conn.rollback()
    print(f"✗ Error: {e}")

## Verificar registros insertados

In [ ]:
# Consultar todos los cursos insertados
sql_select = """
SELECT * FROM cursos
ORDER BY id_curso;
"""

cur.execute(sql_select)
registros = cur.fetchall()

print("Cursos en base de datos:")
print("-" * 60)
for reg in registros:
    print(f"ID: {reg[0]} | Nombre: {reg[1]} | Duración: {reg[2]} | Modalidad: {reg[3]}")

## Cerrar conexión

In [ ]:
cur.close()
conn.close()
print("Conexión cerrada")